# 第3章：异常检测、事件评分与未知诊断

先在仓库根目录安装requirements.txt。按顺序执行；本Notebook可从仓库根目录或本章目录运行。数据为教学构造数据，不代表现场性能。

In [ ]:
from pathlib import Path
import json, csv, sys, tempfile
here=Path.cwd().resolve()
repo=next(p for p in [here,*here.parents] if (p/'chapters/ch03/run.py').exists())
chapter=repo/'chapters/ch03'
sys.path.insert(0,str(chapter))
import run as experiment
cfg=json.loads((chapter/'config.json').read_text(encoding='utf-8'))
raw, events=experiment.generate(cfg)
print('观测数',len(raw),'事件数',len(events))


## 1. 从原始观测构造因果窗口

检查测试设备D16的温升事件。60—87 min电压A缺失；88 min开始恢复，92 min才重新获得完整5点。

In [ ]:
features=experiment.features(raw,cfg['window'])
[(r['minute'],r['valid']) for r in features if r['session_id']=='D16_cooling_loss' and r['minute'] in [59,60,80,87,88,91,92]]


## 2. 重新完成训练、验证选择与冻结测试

使用临时目录保护既有结果。运行结束后比较事件命中、漏报、误报暴露时间与条件延迟。

In [ ]:
output=Path(tempfile.mkdtemp(prefix='ch03_notebook_'))/'experiment'
result=experiment.run(output)
metrics=json.loads((output/'metrics.json').read_text(encoding='utf-8'))
for method in metrics['selected']:
    print(method,metrics['selected'][method],metrics['metrics'][method]['test'])


## 3. 高分为何仍然未知

D13组合异常100 min同时满足电压通道与温升签名。分类器最大分值不是原因已经确定的概率。

In [ ]:
with (output/'test_scores.csv').open(encoding='utf-8',newline='') as f:
    row=next(r for r in csv.DictReader(f) if r['session_id']=='D13_novel_dual' and r['minute']=='100')
{k:row[k] for k in ['channel_difference_v','temperature_residual_c','class_prediction','class_confidence']}


## 4. 打开交互演示并记录一次探索

演示展示冻结参考结果，不自动读取Notebook临时目录。若要替换，显式运行build_demo.py --results <output>。

In [ ]:
from IPython.display import HTML,display
html_path=repo/'demo/ch03/教学演示.html'
display(HTML('<p>请在本地浏览器打开：<code>'+str(html_path)+'</code></p>'))
print('本次独立输出目录：',output)
